# Introducción al OCR con Tesseract
**OCR (Optical Character Recognition / Reconocimiento Óptico de Caracteres)** es la técnica que permite extraer texto legible a partir de una imagen. **Tesseract** es el motor de OCR de código abierto más usado, y `pytesseract` es simplemente un wrapper de Python que lo invoca.

Importante: `pip install pytesseract` (ya agregado a `requirements.txt`) **no alcanza**. Tesseract es un binario externo que se instala a nivel de sistema operativo. Revisá la sección *Requisito extra para el módulo de OCR* en el `README.md` de la raíz del repo para instalarlo según tu SO antes de correr esta lección.

In [ ]:
import cv2  # OpenCV para generar y preprocesar la imagen antes de pasarla al OCR
import numpy as np  # NumPy para crear el lienzo en blanco y generar ruido sintético
import pytesseract  # Wrapper de Python que invoca al binario externo de Tesseract-OCR y parsea su salida

# En Windows, si Tesseract no está en el PATH, descomentá y ajustá esta línea:
# pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# Generar una imagen de muestra con texto
Como no tenemos un asset con texto en `imgs/`, generamos un lienzo blanco con NumPy y escribimos una frase en negro con `cv2.putText`, igual que se usó en módulos anteriores para dibujar sobre frames

In [ ]:
canvas = np.full((200, 600, 3), 255, dtype=np.uint8)  # Crea un lienzo de 200x600 píxeles a color, con todos los píxeles en 255 (blanco): sirve como "hoja en blanco" sintética ya que no hay ninguna imagen con texto en imgs/

cv2.putText(canvas, "Hola OpenCV y Tesseract", (30, 110), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 0), 2)  # Escribe el texto en negro (0, 0, 0) sobre el fondo blanco, con la misma función usada en módulos anteriores para dibujar sobre frames; el alto contraste negro-sobre-blanco es justamente lo que después va a facilitar el OCR

cv2.imshow("Imagen de muestra", canvas)
cv2.waitKey(0)
cv2.destroyAllWindows()

# Preprocesamiento antes del OCR
Convertimos a escala de grises con `cv2.cvtColor` y aplicamos un threshold binario con Otsu (`cv2.THRESH_BINARY + cv2.THRESH_OTSU`). Esto ayuda mucho a Tesseract porque reduce el ruido y deja el texto bien contrastado contra el fondo

In [ ]:
gray = cv2.cvtColor(canvas, cv2.COLOR_BGR2GRAY)  # Convierte a escala de grises: Tesseract trabaja mejor con una imagen de un solo canal, y además el threshold binario que sigue lo requiere

_, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)  # Binariza automáticamente con el método de Otsu (el 0 como umbral se ignora, Otsu calcula el óptimo a partir del histograma). Esto deja el texto en un solo color totalmente contrastado contra el fondo, que es justo el tipo de entrada con el que el motor de Tesseract fue entrenado y funciona mejor: reduce el ruido de antialiasing/gradientes de gris que pueden confundir el reconocimiento de caracteres

cv2.imshow("Imagen preprocesada", thresh)
cv2.waitKey(0)
cv2.destroyAllWindows()

# Aplicar OCR con pytesseract
`pytesseract.image_to_string` recibe una imagen (array de NumPy) y devuelve el texto detectado. Comparamos el resultado sobre la imagen preprocesada contra el resultado sobre la imagen original sin procesar

In [ ]:
texto_preprocesado = pytesseract.image_to_string(thresh)  # Ejecuta Tesseract sobre la imagen ya binarizada (blanco/negro de alto contraste): recibe un array de NumPy y devuelve el texto reconocido como string
print("Texto extraído (con preprocesamiento):")
print(texto_preprocesado)

texto_sin_procesar = pytesseract.image_to_string(canvas)  # Corre el mismo OCR pero sobre la imagen original a color sin binarizar, para comparar: Tesseract internamente también binariza, pero con su propio umbral genérico, que puede ser menos preciso que un Otsu ajustado a esta imagen puntual
print("Texto extraído (sin preprocesamiento):")
print(texto_sin_procesar)

# Variante con rotación: cómo empeora el OCR
Rotamos la imagen original con `cv2.getRotationMatrix2D` y `cv2.warpAffine` (mismo patrón usado en el módulo 1) y volvemos a aplicar OCR **sin** ningún preprocesamiento adicional para corregir la rotación. El resultado suele degradarse notablemente, porque Tesseract espera texto horizontal

In [ ]:
(height, width) = canvas.shape[:2]
center = (width // 2, height // 2)  # Centro de la imagen: punto alrededor del cual se aplica la rotación

rotation_matrix = cv2.getRotationMatrix2D(center, 15, 1.0)  # Genera la matriz de transformación afín para rotar 15 grados alrededor del centro, sin escalar (factor 1.0); mismo patrón usado en el módulo 1 para rotaciones
rotated_canvas = cv2.warpAffine(canvas, rotation_matrix, (width, height), borderValue=(255, 255, 255))  # Aplica la matriz de rotación a la imagen; borderValue=(255,255,255) rellena de blanco las esquinas que quedan vacías tras rotar, para no dejar bordes negros que confundirían al OCR

cv2.imshow("Imagen rotada", rotated_canvas)
cv2.waitKey(0)
cv2.destroyAllWindows()

texto_rotado = pytesseract.image_to_string(rotated_canvas)  # Corre OCR directamente sobre la imagen rotada, sin corregir el ángulo ni binarizar: Tesseract espera texto horizontal, así que la rotación degrada notablemente el reconocimiento
print("Texto extraído (imagen rotada, sin corrección):")
print(texto_rotado)

# Variante con ruido
Agregamos ruido gaussiano simple sobre la imagen original y comparamos cómo afecta al resultado de OCR sin preprocesamiento adicional

In [ ]:
ruido = np.random.normal(0, 25, canvas.shape).astype(np.int16)  # Genera ruido gaussiano aleatorio (media 0, desvío estándar 25) con la misma forma que la imagen; se usa int16 porque el ruido puede ser negativo y uint8 no admite valores negativos
canvas_ruidoso = np.clip(canvas.astype(np.int16) + ruido, 0, 255).astype(np.uint8)  # Suma el ruido a la imagen (convertida a int16 para evitar overflow/underflow durante la suma), recorta el resultado al rango válido [0, 255] con np.clip y vuelve a convertir a uint8 para poder mostrarla/procesarla como imagen normal

cv2.imshow("Imagen con ruido", canvas_ruidoso)
cv2.waitKey(0)
cv2.destroyAllWindows()

texto_ruidoso = pytesseract.image_to_string(canvas_ruidoso)  # OCR sobre la imagen ruidosa sin ningún preprocesamiento adicional: el ruido introduce variaciones de intensidad que dificultan que Tesseract distinga los trazos del texto del fondo
print("Texto extraído (imagen con ruido, sin preprocesamiento):")
print(texto_ruidoso)

## 🧪 Práctica
Reforzá lo aprendido en este módulo resolviendo los ejercicios guiados en [`practicas/11_practica.ipynb`](../practicas/11_practica.ipynb).